In [1]:
# pip install zss
import zss
import json
import ast
# pip install apted astunparse
from apted import APTED, Config

In [2]:
class ASTNode:
    __slots__ = ("n",)
    def __init__(self, n): self.n = n
    # generator of wrapped children
    def children(self):
        for c in ast.iter_child_nodes(self.n):
            yield ASTNode(c)
    # label used for rename-cost
    def label(self):
        if isinstance(self.n, ast.Name):
            return f"Name:{self.n.id}"
        if isinstance(self.n, ast.Constant):
            return f"Const:{self.n.value!r}"
        if isinstance(self.n, ast.operator):
            return type(self.n).__name__
        return type(self.n).__name__


In [3]:
class UnitCost(Config):
    # ---- cost functions ----------------------------------------------------
    def rename(self, a, b): return 0 if a.label()==b.label() else 1
    def insert(self, a):     return 1
    def delete(self, a):     return 1
    # ---- glue: crucial fix -------------------------------------------------
    def children(self, node):
        # MUST return an *iterable*, not the method object itself
        return list(node.children())          # or tuple(node.children())


In [4]:
def code_tree_edit_distance(code1, code2):
    t1 = ASTNode(ast.parse(code1))
    t2 = ASTNode(ast.parse(code2))
    ted = APTED(t1, t2, UnitCost()).compute_edit_distance()
    return ted

In [5]:
# count the number of nodes in the tree
def count_nodes(node):
    return 1 + sum(count_nodes(child) for child in node.children())

In [6]:
code_data = json.load(open('/checkpoint/maui/zhaobc/scientist/code_analysis.json', 'r'))
code_data = {int(k): v for k, v in code_data.items()}

### Directly compare the distance by AST tree editing distance

Reference: 
- Revisiting Code Similarity Evaluation with Abstract Syntax Tree Edit Distance (https://aclanthology.org/2024.acl-short.3.pdf)

The idea is basically that we can take the AST of the ground truth next record (next_human) and compare it with the AI generated code and see if there's difference in the editing distance of the AST for the AI code which reproduces (ai_repro) and the AI code that do not reproduces (ai_negative).

In [7]:
for record_id, data in code_data.items():
    print(f"Record ID: {record_id}")
    gt_code = data['next_human']['code']
    ai_repro_code = data['ai_repro']['code']
    ai_negative_code = data['ai_negative']['code']

    gt_node_count = count_nodes(ASTNode(ast.parse(gt_code)))
    ai_repro_node_count = count_nodes(ASTNode(ast.parse(ai_repro_code)))
    ai_negative_node_count = count_nodes(ASTNode(ast.parse(ai_negative_code)))
    print(f"GT Node Count: {gt_node_count}")
    print(f"AI Repro Node Count: {ai_repro_node_count}")
    print(f"AI Negative Node Count: {ai_negative_node_count}")

    edit_distance_repro = code_tree_edit_distance(gt_code, ai_repro_code)
    edit_distance_negative = code_tree_edit_distance(gt_code, ai_negative_code)

    print(f"Edit Distance (Repro): {edit_distance_repro}")
    print(f"Edit Distance (Negative): {edit_distance_negative}")
    print(f"GT Code Length: {len(gt_code)}")
    print(f"AI Repro Code Length: {len(ai_repro_code)}")
    print(f"AI Negative Code Length: {len(ai_negative_code)}")
    print("\n")
    # takes about 11 minutes to run one record
    break


Record ID: 1
GT Node Count: 3330
AI Repro Node Count: 3435
AI Negative Node Count: 3437


Edit Distance (Repro): 714
Edit Distance (Negative): 714
GT Code Length: 18590
AI Repro Code Length: 19238
AI Negative Code Length: 19251




### Compare the editing script of the positive and negative AI codes

Reference
- Generating Accurate and Compact Edit Scripts using Tree Differencing (https://pinzger.github.io/papers/Frick2018-ijm.pdf)

Another idea is that we can generate the editing script from the record i to record i+1 on GT codes, and then check the editing script of record i to ai_repro and editing script of record i to ai_negative.
An editing script is basically a set of changes that modified code A to code B.
And then we can compute the precision/recall and F1 scores for computing how well the model follows the correct edit.



In [8]:
import itertools
class Node:
    _counter = itertools.count()       # unique id for reproducible hashes
    __slots__ = ("n", "id")
    def __init__(self, n):
        self.n  = n
        self.id = next(Node._counter)  # needed to identify a node position

    def label(self):                   # tune granularity here
        if isinstance(self.n, ast.Name):      return f"Name:{self.n.id}"
        if isinstance(self.n, ast.Constant):  return f"C:{self.n.value!r}"
        if isinstance(self.n, ast.operator):  return type(self.n).__name__
        return type(self.n).__name__

    def children(self):
        for c in ast.iter_child_nodes(self.n):
            yield Node(c)

In [9]:
# uniform cost for insert/delete/rename
class One(Config):
    # make APTED happy
    def children(self, node):     return list(node.children())
    def insert(self, _):     return 1
    def delete(self, _):     return 1
    def rename(self, a, b):  return 0 if a.label()==b.label() else 1

In [10]:
def edits_from(O, X):
    ted = APTED(O, X, One())
    mapping = ted.compute_edit_mapping()      # list of (o_node|None, x_node|None)

    ops = []
    for o, x in mapping:
        if o and x:
            if o.label() != x.label(): ops.append(("upd", o.label(), x.label()))
        elif o and x is None:  ops.append(("del", o.label()))
        elif x and o is None:  ops.append(("ins", x.label()))
    return set(ops)                            # order-free comparison

In [11]:
from math import isnan

def get_edit_form(O_code, G_code, M_code=None):
    # O is the original code
    # G is the ground truth code
    # M is the AI code
    O = Node(ast.parse(O_code))
    G = Node(ast.parse(G_code))
    if M_code is not None:
        M = Node(ast.parse(M_code))
        return edits_from(O, G), edits_from(O, M)
    else:
        return edits_from(O, G)

def prf(gt_edit_form, pr_edit_form):
    gt_ops = gt_edit_form
    pr_ops = pr_edit_form

    tp = len(gt_ops & pr_ops)
    fp = len(pr_ops - gt_ops)
    fn = len(gt_ops - pr_ops)

    prec = tp / (tp + fp) if tp+fp else float("nan")
    rec  = tp / (tp + fn) if tp+fn else float("nan")
    f1   = 2*prec*rec / (prec+rec) if not isnan(prec) and not isnan(rec) else float("nan")
    return prec, rec, f1

In [12]:
for record_id, data in code_data.items():
    o_code = data['human']['code']
    gt_code = data['next_human']['code']
    ai_repro_code = data['ai_repro']['code']
    ai_negative_code = data['ai_negative']['code']

    gt_edit_form = get_edit_form(o_code, gt_code)
    ai_repro_edit_form = get_edit_form(o_code, ai_repro_code)
    ai_negative_edit_form = get_edit_form(o_code, ai_negative_code)

    prec_repro, rec_repro, f1_repro = prf(gt_edit_form, ai_repro_edit_form)
    prec_negative, rec_negative, f1_negative = prf(gt_edit_form, ai_negative_edit_form)

    print(f"Record ID: {record_id}")
    print(f"Precision (Repro): {prec_repro}")
    print(f"Recall (Repro): {rec_repro}")
    print(f"F1 Score (Repro): {f1_repro}")
    print("\n")
    print(f"Precision (Negative): {prec_negative}")
    print(f"Recall (Negative): {rec_negative}")
    print(f"F1 Score (Negative): {f1_negative}")
    print("\n")

    # takes about 20 minutes to run one record
    break

Record ID: 1
Precision (Repro): 0.6837606837606838
Recall (Repro): 0.45714285714285713
F1 Score (Repro): 0.5479452054794521


Precision (Negative): 0.6833333333333333
Recall (Negative): 0.4685714285714286
F1 Score (Negative): 0.5559322033898305


